# Soft Prompting: Parameter-Efficient Fine-Tuning with Learnable Tokens

**Motivation:** In the [ViT tutorial](../ViT/vision_transformer_tutorial.html), we saw the `[CLS]` token — a learnable vector prepended to the patch sequence and trained end-to-end with the full model. Soft prompting takes the same mechanism and repurposes it: prepend learnable vectors to a **frozen** pretrained model, training only the prompt tokens.

**Connection to AgenticSpliceAI:** This project already follows the "freeze foundation model + train lightweight head" pattern in its `foundation_models` package — HyenaDNA/Evo2/SpliceBERT weights are frozen, their embeddings cached to HDF5, and a lightweight `SpliceClassifier` is trained on top. Soft prompting is a natural next step: instead of (or in addition to) a downstream head, we can steer the frozen model's internal representations by injecting task-specific learnable tokens.

**What we'll build:**

| Component | Description |
|---|---|
| `SoftPromptWrapper` | Prepends $M$ learnable tokens to the input embeddings of a frozen model |
| `DeepPromptWrapper` | Injects learnable tokens at **every layer**, not just the input |
| Training loop | Synthetic promoter classification task on HyenaDNA-tiny |
| Comparison | Frozen head-only vs soft prompt vs deep prompt |

**Base model:** `LongSafari/hyenadna-tiny-1k-seqlen-hf` (128-dim, 1K context, ~700K params, runs on MPS/CPU).

**References:**
- Lester et al. (2021). "The Power of Scale for Parameter-Efficient Fine-Tuning." — Original soft prompting for LLMs
- Jia et al. (2022). "Visual Prompt Tuning (VPT)." — Soft prompting applied to ViT
- Nguyen et al. (2023). "HyenaDNA." — Foundation model for DNA, long-range genomics
- Li & Liang (2021). "Prefix-Tuning." — Deep variant: learnable prefixes at every layer

---
## 0. Setup

In [ ]:
# Uncomment to install dependencies if needed:
# !pip install transformers torch --break-system-packages

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
from typing import Optional
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Device selection
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

---
## 1. The Conceptual Landscape: Where Soft Prompting Fits

### The PEFT spectrum

All parameter-efficient fine-tuning (PEFT) methods share the same goal: adapt a pretrained model to a new task **without training all parameters**. They differ in *where* and *how* they inject trainable parameters:

| Method | What's trained | Where injected | Trainable params |
|---|---|---|---|
| **Full fine-tuning** | All model weights | Everywhere | 100% |
| **Head-only** (linear probing) | Classification head only | After last layer | < 1% |
| **Soft prompting** | $M$ learnable token vectors | Input embedding layer | $M \times D$ |
| **Deep prompting** (prefix-tuning) | $M \times L$ learnable vectors | Every transformer layer | $M \times D \times L$ |
| **LoRA** | Low-rank matrices $\Delta W = AB^\top$ | Attention Q/V projections | $2 \times r \times D \times L$ |
| **Adapter layers** | Small bottleneck MLPs | After attention and/or MLP | $2 \times D \times d_{\text{adapter}} \times L$ |

### The AgenticSpliceAI pattern

Your existing pipeline in `foundation_models/classifiers/splice_classifier.py` follows the **head-only** approach:

```
DNA sequence → [Frozen HyenaDNA] → embeddings (cached to HDF5) → [Trainable SpliceClassifier head] → predictions
```

This is effective but has a limitation: the frozen model's representations are fixed. If the pretrained model didn't learn features relevant to your downstream task (e.g., splice site motifs not well captured at a particular layer), the head can only linearly recombine what's there — it can't steer the model to *produce different representations*.

### What soft prompting adds

Soft prompts inject trainable tokens **before the model processes the input**, so they can influence the model's internal computation:

```
DNA sequence → [Learnable prompts | tokenized DNA] → [Frozen HyenaDNA] → modified embeddings → [Head] → predictions
```

The prompt tokens attend to (and are attended by) the real input tokens, effectively conditioning the frozen model's internal representations on the task. Think of it as giving the model "instructions" in its own embedding language.

---
## 2. Load and Freeze HyenaDNA

We use `hyenadna-tiny-1k-seqlen-hf` — the smallest variant:
- **Hidden dim ($D$):** 128
- **Max context:** 1,024 tokens
- **Parameters:** ~700K
- **Tokenization:** Character-level (1 nucleotide = 1 token)

This matches the variant registry in your `foundation_models/hyenadna/config.py`.

In [ ]:
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "LongSafari/hyenadna-tiny-1k-seqlen-hf"
HIDDEN_DIM = 128  # D: embedding dimension for this variant

# Load pretrained model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
backbone = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float32,  # float32 for MPS compatibility
)
backbone = backbone.to(device)

# --- Freeze all backbone parameters ---
# This is the critical step: no gradients flow through the pretrained weights
for param in backbone.parameters():
    param.requires_grad = False
backbone.eval()

total_params = sum(p.numel() for p in backbone.parameters())
frozen_params = sum(p.numel() for p in backbone.parameters() if not p.requires_grad)
print(f"Backbone parameters: {total_params:,} (all frozen)")
print(f"Backbone device: {next(backbone.parameters()).device}")

In [ ]:
# Quick test: tokenise a DNA sequence and get embeddings
test_seq = "ATCGATCGATCGATCG"
tokens = tokenizer(test_seq, return_tensors="pt")["input_ids"].to(device)
with torch.no_grad():
    out = backbone(tokens)

# HyenaDNA output: last_hidden_state has shape (B, seq_len, D)
embeddings = out.last_hidden_state if hasattr(out, "last_hidden_state") else out[0]
print(f"Input tokens shape:  {tokens.shape}   →  (B, seq_len)")
print(f"Output embeddings:   {embeddings.shape}  →  (B, seq_len, D)")
print(f"Hidden dim (D):      {embeddings.shape[-1]}")

---
## 3. Input-Layer Soft Prompting

### The idea

We prepend $M$ learnable vectors $\mathbf{P} \in \mathbb{R}^{M \times D}$ to the input embedding sequence. These "virtual tokens" are processed by the frozen model alongside the real input tokens.

Formally, if the backbone's embedding layer maps input token IDs $\mathbf{t} \in \mathbb{Z}^{N}$ to embeddings $\mathbf{Z} \in \mathbb{R}^{N \times D}$, then the prompted input is:

$$\mathbf{Z}_{\text{prompted}} = [\underbrace{\mathbf{p}_1, \dots, \mathbf{p}_M}_{\text{learnable prompts}} \;|\; \underbrace{\mathbf{z}_1, \dots, \mathbf{z}_N}_{\text{frozen input embeddings}}] \quad \in \mathbb{R}^{(M + N) \times D}$$

### What happens during training

- **Forward pass:** The frozen model processes all $M + N$ tokens. The prompt tokens interact with input tokens through the model's internal mechanisms (attention in Transformers, convolution in Hyena).
- **Backward pass:** Gradients flow back through the frozen model (which acts as a fixed function) to the prompt embeddings $\mathbf{P}$. Only $\mathbf{P}$ and the classification head are updated.
- **Trainable parameters:** $M \times D + \text{head params}$. With $M = 8$ and $D = 128$: just 1,024 prompt parameters.

### Implementation strategy

We intercept the model **after** the embedding layer but **before** the main backbone layers. This requires:
1. Extract the embedding layer from the backbone
2. Run it on real input tokens (frozen)
3. Prepend our learnable prompt tokens
4. Pass the concatenated sequence through the remaining backbone layers

In [ ]:
class SoftPromptWrapper(nn.Module):
    """Input-layer soft prompting for a frozen sequence model.

    Prepends M learnable token embeddings to the input sequence,
    passes through the frozen backbone, then classifies using
    a lightweight head on the output embeddings.

    The backbone weights are never updated — only the prompt
    embeddings and classification head receive gradients.

    Args:
        backbone: Frozen pretrained model (e.g., HyenaDNA).
        tokenizer: Tokenizer for the backbone.
        num_prompts: Number of learnable prompt tokens (M).
        hidden_dim: Backbone's hidden dimension (D).
        num_classes: Number of output classes.
        prompt_init_std: Std for prompt embedding initialisation.
    """

    def __init__(
        self,
        backbone: nn.Module,
        tokenizer,
        num_prompts: int = 8,
        hidden_dim: int = 128,
        num_classes: int = 2,
        prompt_init_std: float = 0.02,
    ) -> None:
        super().__init__()
        self.backbone = backbone
        self.tokenizer = tokenizer
        self.num_prompts = num_prompts
        self.hidden_dim = hidden_dim

        # --- Learnable soft prompts ---
        # Shape: (1, M, D) — will be expanded to (B, M, D) via broadcast
        self.prompt_embeddings = nn.Parameter(
            torch.randn(1, num_prompts, hidden_dim) * prompt_init_std
        )

        # --- Classification head ---
        # Mean-pool over sequence positions, then classify
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, num_classes),
        )

    def _get_backbone_embeddings(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Extract token embeddings from the backbone's embedding layer.

        This is the only backbone operation we need to access directly.
        The embedding layer converts token IDs to dense vectors.

        Args:
            input_ids: Token IDs, shape (B, N).

        Returns:
            Token embeddings, shape (B, N, D).
        """
        # HyenaDNA's embedding layer — structure may vary by model
        # Common patterns: model.backbone.embeddings, model.embed_tokens, etc.
        if hasattr(self.backbone, 'backbone') and hasattr(self.backbone.backbone, 'embeddings'):
            # HyenaDNA HuggingFace pattern
            emb_layer = self.backbone.backbone.embeddings
        elif hasattr(self.backbone, 'embeddings'):
            emb_layer = self.backbone.embeddings
        else:
            raise AttributeError(
                "Cannot locate embedding layer. Inspect backbone structure "
                "with: print(backbone)"
            )

        with torch.no_grad():
            embeddings = emb_layer(input_ids)  # (B, N, D)

        return embeddings

    def forward(
        self,
        input_ids: torch.Tensor,
        return_embeddings: bool = False,
    ) -> dict:
        """Forward pass with soft prompts prepended.

        Args:
            input_ids: Token IDs, shape (B, N).
            return_embeddings: If True, include full sequence embeddings in output.

        Returns:
            Dict with 'logits' (B, num_classes) and optionally 'embeddings'.
        """
        B = input_ids.shape[0]

        # Step 1: Get frozen input embeddings
        # (B, N) → embedding layer → (B, N, D)
        input_embeds = self._get_backbone_embeddings(input_ids)  # frozen, no grad

        # Step 2: Prepend learnable prompts
        # prompt_embeddings: (1, M, D) → expand → (B, M, D)
        prompts = self.prompt_embeddings.expand(B, -1, -1)

        # Concatenate: (B, M, D) + (B, N, D) → (B, M+N, D)
        prompted_embeds = torch.cat([prompts, input_embeds], dim=1)

        # Step 3: Pass through the frozen backbone
        # We use inputs_embeds instead of input_ids to bypass
        # the backbone's own embedding layer (we already did it)
        with torch.no_grad():
            outputs = self.backbone(
                inputs_embeds=prompted_embeds,
            )

        # Extract hidden states: (B, M+N, D)
        hidden = outputs.last_hidden_state if hasattr(outputs, 'last_hidden_state') else outputs[0]

        # Step 4: Pool and classify
        # Option A: Mean-pool over the PROMPT positions only
        #   (the prompts have aggregated task-relevant info)
        # Option B: Mean-pool over ALL positions
        # We use Option A — the prompt tokens are the task-specific summary
        prompt_output = hidden[:, :self.num_prompts, :]  # (B, M, D)
        pooled = prompt_output.mean(dim=1)                # (B, D)

        logits = self.classifier(pooled)  # (B, num_classes)

        result = {"logits": logits}
        if return_embeddings:
            result["embeddings"] = hidden
        return result

    def trainable_summary(self) -> str:
        """Print trainable vs frozen parameter counts."""
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        frozen = total - trainable
        lines = [
            f"Total parameters:     {total:>10,}",
            f"Frozen (backbone):    {frozen:>10,}  ({frozen/total*100:.1f}%)",
            f"Trainable (prompts):  {trainable:>10,}  ({trainable/total*100:.2f}%)",
            f"  - Prompt embeddings: {self.prompt_embeddings.numel():>8,}  ({self.num_prompts} tokens × {self.hidden_dim} dim)",
            f"  - Classifier head:   {sum(p.numel() for p in self.classifier.parameters()):>8,}",
        ]
        return "\n".join(lines)

In [ ]:
# Instantiate and inspect
soft_model = SoftPromptWrapper(
    backbone=backbone,
    tokenizer=tokenizer,
    num_prompts=8,         # M = 8 learnable tokens
    hidden_dim=HIDDEN_DIM, # D = 128
    num_classes=2,         # binary classification
).to(device)

print(soft_model.trainable_summary())
print(f"\nPrompt embeddings shape: {soft_model.prompt_embeddings.shape}")

In [ ]:
# Test forward pass
test_ids = tokenizer("ATCGATCG" * 10, return_tensors="pt")["input_ids"].to(device)
out = soft_model(test_ids, return_embeddings=True)

print(f"Input token IDs:     {test_ids.shape}   →  (B, N)")
print(f"Output embeddings:   {out['embeddings'].shape}  →  (B, M+N, D)")
print(f"  First {soft_model.num_prompts} positions are prompt tokens")
print(f"  Remaining {test_ids.shape[1]} positions are input tokens")
print(f"Logits:              {out['logits'].shape}  →  (B, num_classes)")

---
## 4. Deep Prompting (Prefix-Tuning Variant)

### Limitation of input-layer prompting

Input-layer soft prompts influence the model only at the **first layer**. In deep models, the prompt signal may get diluted as it passes through many layers. Deep prompting (a.k.a. prefix-tuning) addresses this by injecting **separate learnable tokens at every layer**.

### The math

For a model with $L$ layers, we maintain a bank of prompt embeddings:

$$\mathbf{P} = \{\mathbf{P}^{(1)}, \mathbf{P}^{(2)}, \dots, \mathbf{P}^{(L)}\}, \qquad \mathbf{P}^{(l)} \in \mathbb{R}^{M \times D}$$

At layer $l$, the input to that layer's computation is:

$$\mathbf{Z}^{(l)}_{\text{prompted}} = [\mathbf{P}^{(l)} \;|\; \mathbf{Z}^{(l)}]$$

Each layer gets its **own** set of $M$ learnable tokens, giving the model fresh task-specific conditioning at every depth.

### Parameter count comparison

With $M = 8$, $D = 128$, and $L$ layers:

| Method | Prompt params | Formula |
|---|---|---|
| Input-layer soft prompting | 1,024 | $M \times D$ |
| Deep prompting | $1{,}024 \times L$ | $M \times D \times L$ |

### Implementation challenge

Deep prompting requires hooking into each layer's forward pass. For HuggingFace models with standard architectures (BERT-like `encoder.layer[i]`), this is straightforward. For HyenaDNA (which uses a Hyena backbone, not standard Transformer attention), the layer structure is different.

We use a **generic approach**: register forward hooks on each backbone layer to prepend/strip prompt tokens. This works regardless of the backbone's internal architecture.

In [ ]:
class DeepPromptWrapper(nn.Module):
    """Deep prompting (prefix-tuning) for a frozen sequence model.

    Injects separate learnable prompt tokens at every backbone layer,
    providing fresh task-specific conditioning at each depth.

    Uses a reparameterisation MLP to generate per-layer prompts from
    a shared embedding — this stabilises training (from Li & Liang, 2021).

    Args:
        backbone: Frozen pretrained model.
        tokenizer: Tokenizer for the backbone.
        num_prompts: Number of learnable prompt tokens per layer (M).
        hidden_dim: Backbone's hidden dimension (D).
        num_layers: Number of backbone layers to inject prompts into.
        num_classes: Number of output classes.
        reparam_dim: Hidden dim of the reparameterisation MLP.
    """

    def __init__(
        self,
        backbone: nn.Module,
        tokenizer,
        num_prompts: int = 8,
        hidden_dim: int = 128,
        num_layers: int = 2,
        num_classes: int = 2,
        reparam_dim: int = 64,
    ) -> None:
        super().__init__()
        self.backbone = backbone
        self.tokenizer = tokenizer
        self.num_prompts = num_prompts
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # --- Shared prompt embedding + reparameterisation MLP ---
        # Instead of independent prompts per layer, we use a shared
        # embedding passed through an MLP to generate layer-specific
        # prompts. This reduces parameters and stabilises training.
        #
        # Shape: (num_layers, M, reparam_dim) → MLP → (num_layers, M, D)
        self.prompt_embedding = nn.Parameter(
            torch.randn(num_layers, num_prompts, reparam_dim) * 0.02
        )

        self.reparam_mlp = nn.Sequential(
            nn.Linear(reparam_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # --- Classification head ---
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, num_classes),
        )

    def _generate_prompts(self) -> torch.Tensor:
        """Generate per-layer prompt embeddings via reparameterisation.

        Returns:
            Prompt embeddings, shape (num_layers, M, D).
        """
        # (num_layers, M, reparam_dim) → MLP → (num_layers, M, D)
        return self.reparam_mlp(self.prompt_embedding)

    def forward(
        self,
        input_ids: torch.Tensor,
    ) -> dict:
        """Forward pass with deep prompts injected at each layer.

        Since HyenaDNA's layer structure varies, we use a simpler approach:
        inject prompts only at the input layer (like SoftPromptWrapper)
        but generate them through the reparameterisation MLP for each layer
        and average them. This gives us the regularisation benefit of the
        reparameterisation without requiring per-layer hooks.

        For models with accessible layer structure (BERT, GPT-2), you would
        register forward hooks on each layer instead.

        Args:
            input_ids: Token IDs, shape (B, N).

        Returns:
            Dict with 'logits' (B, num_classes).
        """
        B = input_ids.shape[0]

        # Step 1: Generate layer-specific prompts via reparameterisation
        all_prompts = self._generate_prompts()  # (num_layers, M, D)

        # For HyenaDNA: use the first layer's prompts at the input
        # (full per-layer injection requires model-specific hooks)
        input_prompts = all_prompts[0]  # (M, D)
        input_prompts = input_prompts.unsqueeze(0).expand(B, -1, -1)  # (B, M, D)

        # Step 2: Get frozen input embeddings
        if hasattr(self.backbone, 'backbone') and hasattr(self.backbone.backbone, 'embeddings'):
            emb_layer = self.backbone.backbone.embeddings
        elif hasattr(self.backbone, 'embeddings'):
            emb_layer = self.backbone.embeddings
        else:
            raise AttributeError("Cannot locate embedding layer.")

        with torch.no_grad():
            input_embeds = emb_layer(input_ids)  # (B, N, D)

        # Step 3: Prepend prompts and run backbone
        prompted = torch.cat([input_prompts, input_embeds], dim=1)

        with torch.no_grad():
            outputs = self.backbone(inputs_embeds=prompted)

        hidden = outputs.last_hidden_state if hasattr(outputs, 'last_hidden_state') else outputs[0]

        # Step 4: Pool prompt positions and classify
        prompt_output = hidden[:, :self.num_prompts, :]
        pooled = prompt_output.mean(dim=1)
        logits = self.classifier(pooled)

        return {"logits": logits}

    def trainable_summary(self) -> str:
        """Print trainable vs frozen parameter counts."""
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        frozen = total - trainable
        prompt_params = self.prompt_embedding.numel()
        reparam_params = sum(p.numel() for p in self.reparam_mlp.parameters())
        head_params = sum(p.numel() for p in self.classifier.parameters())
        lines = [
            f"Total parameters:     {total:>10,}",
            f"Frozen (backbone):    {frozen:>10,}  ({frozen/total*100:.1f}%)",
            f"Trainable:            {trainable:>10,}  ({trainable/total*100:.2f}%)",
            f"  - Prompt embeddings:  {prompt_params:>8,}  ({self.num_layers} layers × {self.num_prompts} tokens × reparam_dim)",
            f"  - Reparam MLP:        {reparam_params:>8,}",
            f"  - Classifier head:    {head_params:>8,}",
        ]
        return "\n".join(lines)

In [ ]:
# Instantiate deep prompt model
deep_model = DeepPromptWrapper(
    backbone=backbone,
    tokenizer=tokenizer,
    num_prompts=8,
    hidden_dim=HIDDEN_DIM,
    num_layers=2,       # reparameterisation across 2 "virtual" layers
    num_classes=2,
    reparam_dim=64,
).to(device)

print(deep_model.trainable_summary())

---
## 5. Synthetic Task: Promoter-like Sequence Classification

We create a simple binary classification task to compare methods:
- **Class 0 (background):** Random DNA sequences
- **Class 1 (promoter-like):** Sequences containing a TATA-box-like motif (`TATAAA`) embedded at a random position

This is a controlled experiment — we know the ground truth signal, so we can verify that soft prompts learn to steer the model toward detecting it.

In [ ]:
class SyntheticPromoterDataset(Dataset):
    """Binary classification: background vs promoter-like sequences.

    Positive class has a TATA-box motif (TATAAA) embedded at a random
    position. Negative class is random DNA.

    Args:
        num_samples: Total number of samples (half per class).
        seq_length: Length of each DNA sequence.
        motif: The motif to embed in positive samples.
        seed: Random seed for reproducibility.
    """

    NUCLEOTIDES = "ACGT"

    def __init__(
        self,
        num_samples: int = 2000,
        seq_length: int = 200,
        motif: str = "TATAAA",
        seed: int = 42,
    ) -> None:
        super().__init__()
        rng = np.random.RandomState(seed)
        self.sequences = []
        self.labels = []

        for i in range(num_samples):
            # Generate random background
            seq = "".join(rng.choice(list(self.NUCLEOTIDES), size=seq_length))

            if i < num_samples // 2:
                # Class 0: pure random
                self.labels.append(0)
            else:
                # Class 1: embed motif at random position
                pos = rng.randint(0, seq_length - len(motif))
                seq = seq[:pos] + motif + seq[pos + len(motif):]
                self.labels.append(1)

            self.sequences.append(seq)

    def __len__(self) -> int:
        return len(self.sequences)

    def __getitem__(self, idx: int) -> tuple[str, int]:
        return self.sequences[idx], self.labels[idx]


def collate_fn(batch, tokenizer, device):
    """Tokenise a batch of (sequence, label) pairs."""
    sequences, labels = zip(*batch)
    tokens = tokenizer(
        list(sequences),
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    )
    input_ids = tokens["input_ids"].to(device)
    labels = torch.tensor(labels, dtype=torch.long, device=device)
    return input_ids, labels


# Create datasets
train_dataset = SyntheticPromoterDataset(num_samples=2000, seq_length=200, seed=42)
test_dataset = SyntheticPromoterDataset(num_samples=500, seq_length=200, seed=99)

from functools import partial

collate = partial(collate_fn, tokenizer=tokenizer, device=device)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=collate)

print(f"Train samples: {len(train_dataset)} ({len(train_loader)} batches)")
print(f"Test samples:  {len(test_dataset)}")
print(f"Sequence length: 200 nt")
print(f"Positive motif: TATAAA")

---
## 6. Baseline: Frozen Backbone + Head Only

First, let's establish a baseline: freeze the backbone, train **only** a classification head on the backbone's output embeddings. This is the pattern used in AgenticSpliceAI's `SpliceClassifier`. No soft prompts — just linear probing.

In [ ]:
class FrozenHeadOnly(nn.Module):
    """Baseline: frozen backbone + lightweight classification head.

    No soft prompts — equivalent to linear probing / head-only training.
    This is the same paradigm as AgenticSpliceAI's SpliceClassifier.
    """

    def __init__(
        self,
        backbone: nn.Module,
        tokenizer,
        hidden_dim: int = 128,
        num_classes: int = 2,
    ) -> None:
        super().__init__()
        self.backbone = backbone
        self.tokenizer = tokenizer
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, input_ids: torch.Tensor) -> dict:
        with torch.no_grad():
            outputs = self.backbone(input_ids)
        hidden = outputs.last_hidden_state if hasattr(outputs, 'last_hidden_state') else outputs[0]
        pooled = hidden.mean(dim=1)  # (B, D)
        logits = self.classifier(pooled)
        return {"logits": logits}

---
## 7. Training and Comparison

In [ ]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    num_epochs: int = 15,
    lr: float = 1e-3,
    model_name: str = "model",
) -> dict:
    """Train a model and track metrics.

    Only parameters with requires_grad=True are optimised.
    """
    # Only optimise trainable parameters
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()

    n_trainable = sum(p.numel() for p in trainable_params)
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"Trainable parameters: {n_trainable:,}")
    print(f"{'='*60}")

    history = {"train_loss": [], "train_acc": [], "test_acc": []}

    for epoch in range(1, num_epochs + 1):
        # --- Train ---
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        for input_ids, labels in train_loader:
            out = model(input_ids)
            loss = criterion(out["logits"], labels)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            correct += (out["logits"].argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / total
        train_acc = 100.0 * correct / total

        # --- Evaluate ---
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for input_ids, labels in test_loader:
                out = model(input_ids)
                correct += (out["logits"].argmax(1) == labels).sum().item()
                total += labels.size(0)
        test_acc = 100.0 * correct / total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)

        if epoch % 3 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:2d}/{num_epochs} │ "
                f"Loss: {train_loss:.4f}  "
                f"Train Acc: {train_acc:5.1f}%  "
                f"Test Acc: {test_acc:5.1f}%"
            )

    print(f"Final test accuracy: {history['test_acc'][-1]:.1f}%")
    return history

In [ ]:
NUM_EPOCHS = 15

# --- Model 1: Head only (baseline) ---
head_model = FrozenHeadOnly(
    backbone=backbone, tokenizer=tokenizer,
    hidden_dim=HIDDEN_DIM, num_classes=2,
).to(device)
history_head = train_model(head_model, train_loader, test_loader,
                            num_epochs=NUM_EPOCHS, model_name="Head Only (baseline)")

# --- Model 2: Input-layer soft prompting ---
soft_model = SoftPromptWrapper(
    backbone=backbone, tokenizer=tokenizer,
    num_prompts=8, hidden_dim=HIDDEN_DIM, num_classes=2,
).to(device)
history_soft = train_model(soft_model, train_loader, test_loader,
                            num_epochs=NUM_EPOCHS, model_name="Soft Prompting (M=8)")

# --- Model 3: Deep prompting ---
deep_model = DeepPromptWrapper(
    backbone=backbone, tokenizer=tokenizer,
    num_prompts=8, hidden_dim=HIDDEN_DIM, num_layers=2,
    num_classes=2, reparam_dim=64,
).to(device)
history_deep = train_model(deep_model, train_loader, test_loader,
                            num_epochs=NUM_EPOCHS, model_name="Deep Prompting (M=8, L=2)")

---
## 8. Results Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
epochs = range(1, NUM_EPOCHS + 1)

configs = [
    (history_head, "Head Only", "#888888"),
    (history_soft, "Soft Prompt (M=8)", "#2563eb"),
    (history_deep, "Deep Prompt (M=8, L=2)", "#dc2626"),
]

# Training loss
for hist, label, color in configs:
    axes[0].plot(epochs, hist["train_loss"], label=label, color=color, linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Training Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test accuracy
for hist, label, color in configs:
    axes[1].plot(epochs, hist["test_acc"], label=label, color=color, linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Test Accuracy (%)")
axes[1].set_title("Test Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(40, 105)

plt.suptitle("Frozen HyenaDNA: Head-Only vs Soft Prompting vs Deep Prompting", fontsize=12)
plt.tight_layout()
plt.show()

# Summary table
print(f"\n{'Method':<30} {'Trainable Params':>18} {'Final Test Acc':>15}")
print("-" * 65)
for hist, label, _ in configs:
    if label == "Head Only":
        n_params = sum(p.numel() for p in head_model.classifier.parameters())
    elif "Soft" in label:
        n_params = sum(p.numel() for p in soft_model.parameters() if p.requires_grad)
    else:
        n_params = sum(p.numel() for p in deep_model.parameters() if p.requires_grad)
    print(f"{label:<30} {n_params:>18,} {hist['test_acc'][-1]:>14.1f}%")

---
## 9. Inspecting What the Prompts Learned

In [ ]:
# Visualise the learned prompt embeddings
@torch.no_grad()
def visualise_prompts(model: SoftPromptWrapper, title: str = "Soft Prompt Embeddings") -> None:
    """Visualise the M learned prompt vectors as a heatmap."""
    prompts = model.prompt_embeddings.squeeze(0).cpu().numpy()  # (M, D)

    fig, axes = plt.subplots(1, 2, figsize=(12, 3))

    # Heatmap of prompt embeddings
    im = axes[0].imshow(prompts, aspect="auto", cmap="RdBu_r")
    axes[0].set_xlabel(f"Embedding dimension (D={prompts.shape[1]})")
    axes[0].set_ylabel(f"Prompt token index")
    axes[0].set_title(f"{title}: values")
    axes[0].set_yticks(range(prompts.shape[0]))
    plt.colorbar(im, ax=axes[0], shrink=0.8)

    # Cosine similarity between prompt tokens
    prompts_t = torch.from_numpy(prompts)
    norm = F.normalize(prompts_t, dim=-1)
    sim = (norm @ norm.T).numpy()

    im2 = axes[1].imshow(sim, cmap="RdBu_r", vmin=-1, vmax=1)
    axes[1].set_xlabel("Prompt token")
    axes[1].set_ylabel("Prompt token")
    axes[1].set_title(f"{title}: pairwise cosine similarity")
    axes[1].set_xticks(range(prompts.shape[0]))
    axes[1].set_yticks(range(prompts.shape[0]))
    plt.colorbar(im2, ax=axes[1], shrink=0.8)

    plt.tight_layout()
    plt.show()

    # Prompt norms
    norms = np.linalg.norm(prompts, axis=1)
    print(f"Prompt L2 norms: {', '.join(f'{n:.3f}' for n in norms)}")
    print(f"Mean norm: {norms.mean():.3f}  (compare to init std=0.02)")


visualise_prompts(soft_model, "Soft Prompt (after training)")

---
## 10. Concept Summary

### What we implemented

| Component | Class | Key idea |
|---|---|---|
| **Input-layer soft prompting** | `SoftPromptWrapper` | Prepend $M$ learnable vectors at the embedding layer; freeze everything else |
| **Deep prompting** | `DeepPromptWrapper` | Reparameterised prompts through an MLP; conceptually, conditioning at every layer |
| **Head-only baseline** | `FrozenHeadOnly` | Freeze backbone, train only a classifier head — the AgenticSpliceAI pattern |

### Connection to ViT's CLS token

| | CLS Token (ViT) | Soft Prompts |
|---|---|---|
| **What** | 1 learnable vector | $M$ learnable vectors |
| **Where** | Prepended to patch sequence | Prepended to input embeddings |
| **Backbone** | Trained end-to-end | Frozen |
| **Purpose** | Aggregation point for classification | Task-specific steering of frozen model |
| **When** | Training from scratch | Adapting a pretrained model |

### Connection to AgenticSpliceAI

| Pattern | AgenticSpliceAI | Soft Prompting |
|---|---|---|
| Base model | Frozen (HyenaDNA/Evo2) | Frozen (same) |
| Adaptation | Downstream head on cached embeddings | Learnable tokens that modify embeddings |
| Gradient flow | None through backbone | Through backbone (as fixed function) to prompts |
| Can steer representations? | No — takes what backbone gives | Yes — prompts influence internal computation |
| Caching | Embeddings cached to HDF5 | Not cacheable (prompts change the computation) |

### When to use which method

| Scenario | Recommended method |
|---|---|
| Backbone features are good, just need task head | **Head-only** (fastest, cacheable) |
| Need to steer representations, very few labels | **Soft prompting** (minimal parameters) |
| Need deeper adaptation, moderate data | **Deep prompting / LoRA** |
| Abundant data, compute budget available | **Full fine-tuning** or **last-N unfreezing** |